In [0]:
# Import the required Delta Lake, Python and PySpark components

from datetime import datetime

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    round as spark_round,
    sum as spark_sum
)

In [0]:
# Receive pipeline parameters and define table names

dbutils.widgets.text(
    "batch_id",
    "2009-12",
    "Batch ID"
)

dbutils.widgets.text(
    "run_id",
    "manual-run-001",
    "Run ID"
)

batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")

bronze_table = "online_retail.bronze.transactions_raw"
silver_table = "online_retail.silver.transactions_clean"
gold_table = "online_retail.gold.product_sales_summary"
control_table = "online_retail.control.pipeline_runs"
layer_name = "validation"

print(f"Run ID: {run_id}")
print(f"Batch ID: {batch_id}")
print(f"Bronze table: {bronze_table}")
print(f"Silver table: {silver_table}")
print(f"Gold table: {gold_table}")

Run ID: validation-cleanup-test-001
Batch ID: 2010-02
Bronze table: online_retail.bronze.transactions_raw
Silver table: online_retail.silver.transactions_clean
Gold table: online_retail.gold.product_sales_summary


In [0]:
# Validate that batch_id is a valid month in exact YYYY-MM format

try:
    batch_month = datetime.strptime(batch_id, "%Y-%m")

    if batch_month.strftime("%Y-%m") != batch_id:
        raise ValueError

except ValueError as error:
    raise ValueError(
        f"Invalid batch_id: {batch_id}. Expected YYYY-MM."
    ) from error

else:
    print(f"Valid batch ID: {batch_id}")

Valid batch ID: 2010-02


In [0]:
# Record that pipeline validation has started

if not spark.catalog.tableExists(control_table):
    raise ValueError(
        f"Control table does not exist: {control_table}"
    )


started_audit_df = (
    spark.range(1)
    .select(
        lit(run_id).alias("run_id"),
        lit(batch_id).alias("batch_id"),
        lit(layer_name).alias("layer_name"),
        lit("STARTED").alias("status"),
        current_timestamp().alias("start_timestamp"),
        lit(None).cast("timestamp").alias("end_timestamp"),
        lit(None).cast("long").alias("input_row_count"),
        lit(None).cast("long").alias("output_row_count"),
        lit(None).cast("string").alias("error_message")
    )
)


control_delta_table = DeltaTable.forName(
    spark,
    control_table
)


(
    control_delta_table.alias("target")
    .merge(
        started_audit_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.batch_id = source.batch_id
        AND target.layer_name = source.layer_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    f"Pipeline validation started for batch {batch_id}, "
    f"run {run_id}."
)

Pipeline validation started for batch 2010-02, run validation-cleanup-test-001.


In [0]:
# Confirm that all required pipeline tables exist

required_tables = [
    bronze_table,
    silver_table,
    gold_table
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]


if missing_tables:
    raise ValueError(
        f"Required pipeline tables are missing: "
        f"{', '.join(missing_tables)}"
    )


# Read the selected batch from every Medallion layer

bronze_batch_df = (
    spark.table(bronze_table)
    .filter(col("batch_id") == batch_id)
)

silver_batch_df = (
    spark.table(silver_table)
    .filter(col("batch_id") == batch_id)
)

gold_batch_df = (
    spark.table(gold_table)
    .filter(col("batch_id") == batch_id)
)


bronze_row_count = bronze_batch_df.count()
silver_row_count = silver_batch_df.count()
gold_row_count = gold_batch_df.count()


print(f"Bronze rows: {bronze_row_count:,}")
print(f"Silver rows: {silver_row_count:,}")
print(f"Gold rows: {gold_row_count:,}")


if bronze_row_count == 0:
    raise ValueError(
        f"Bronze contains no records for batch {batch_id}."
    )

if silver_row_count == 0:
    raise ValueError(
        f"Silver contains no records for batch {batch_id}."
    )

if gold_row_count == 0:
    raise ValueError(
        f"Gold contains no records for batch {batch_id}."
    )

print(
    f"All pipeline layers contain batch {batch_id}."
)

Bronze rows: 29,388
Silver rows: 29,058
Gold rows: 2,577
All pipeline layers contain batch 2010-02.


In [0]:
# Calculate the Silver count expected after business deduplication

business_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country"
]


expected_silver_row_count = (
    bronze_batch_df
    .select(*business_columns)
    .dropDuplicates()
    .count()
)


duplicate_business_row_count = (
    bronze_row_count
    - expected_silver_row_count
)


# Confirm that every Silver record originated in Bronze

orphan_silver_record_count = (
    silver_batch_df
    .select("record_id")
    .join(
        bronze_batch_df.select("record_id"),
        on="record_id",
        how="left_anti"
    )
    .count()
)


print(f"Bronze rows: {bronze_row_count:,}")
print(
    f"Duplicate business rows: "
    f"{duplicate_business_row_count:,}"
)
print(
    f"Expected Silver rows: "
    f"{expected_silver_row_count:,}"
)
print(f"Actual Silver rows: {silver_row_count:,}")
print(
    f"Orphan Silver records: "
    f"{orphan_silver_record_count:,}"
)


if silver_row_count != expected_silver_row_count:
    raise ValueError(
        f"Expected {expected_silver_row_count:,} Silver "
        f"rows but found {silver_row_count:,} for "
        f"batch {batch_id}."
    )

if orphan_silver_record_count > 0:
    raise ValueError(
        f"Found {orphan_silver_record_count} Silver records "
        f"without matching Bronze record IDs."
    )

print(
    f"Bronze-to-Silver reconciliation passed "
    f"for batch {batch_id}."
)

Bronze rows: 29,388
Duplicate business rows: 330
Expected Silver rows: 29,058
Actual Silver rows: 29,058
Orphan Silver records: 0
Bronze-to-Silver reconciliation passed for batch 2010-02.


In [0]:
# Reconcile positive-sale totals between Silver and Gold

positive_sales_batch_df = (
    silver_batch_df
    .filter(col("is_positive_sale"))
)


silver_totals = (
    positive_sales_batch_df
    .agg(
        spark_sum("quantity").alias(
            "silver_total_quantity"
        ),

        spark_round(
            spark_sum("line_total"),
            2
        ).alias("silver_total_revenue")
    )
    .first()
)


gold_totals = (
    gold_batch_df
    .agg(
        spark_sum("total_quantity_sold").alias(
            "gold_total_quantity"
        ),

        spark_round(
            spark_sum("total_revenue"),
            2
        ).alias("gold_total_revenue")
    )
    .first()
)


silver_total_quantity = (
    silver_totals["silver_total_quantity"]
)

silver_total_revenue = (
    silver_totals["silver_total_revenue"]
)

gold_total_quantity = (
    gold_totals["gold_total_quantity"]
)

gold_total_revenue = (
    gold_totals["gold_total_revenue"]
)


quantities_match = (
    silver_total_quantity == gold_total_quantity
)

revenues_match = (
    abs(silver_total_revenue - gold_total_revenue)
    <= 0.01
)


print(
    f"Silver total quantity: "
    f"{silver_total_quantity:,}"
)

print(
    f"Gold total quantity: "
    f"{gold_total_quantity:,}"
)

print(f"Quantities match: {quantities_match}")

print(
    f"Silver total revenue: "
    f"{silver_total_revenue:,.2f}"
)

print(
    f"Gold total revenue: "
    f"{gold_total_revenue:,.2f}"
)

print(f"Revenues match: {revenues_match}")


if not quantities_match:
    raise ValueError(
        f"Silver and Gold quantities do not match "
        f"for batch {batch_id}."
    )

if not revenues_match:
    raise ValueError(
        f"Silver and Gold revenues do not match "
        f"for batch {batch_id}."
    )

print(
    f"Silver-to-Gold reconciliation passed "
    f"for batch {batch_id}."
)

Silver total quantity: 381,879
Gold total quantity: 381,879
Quantities match: True
Silver total revenue: 551,504.72
Gold total revenue: 551,504.72
Revenues match: True
Silver-to-Gold reconciliation passed for batch 2010-02.


In [0]:
# Mark pipeline validation as successful in the control table

control_delta_table.update(
    condition=(
        (col("run_id") == run_id)
        & (col("batch_id") == batch_id)
        & (col("layer_name") == layer_name)
    ),
    set={
        "status": lit("SUCCESS"),
        "end_timestamp": current_timestamp(),
        "input_row_count": lit(bronze_row_count),
        "output_row_count": lit(gold_row_count),
        "error_message": lit(None).cast("string")
    }
)

print(
    f"Audit completed successfully for validation, "
    f"batch {batch_id}, run {run_id}."
)

Audit completed successfully for validation, batch 2010-02, run validation-cleanup-test-001.
